# Get data from Google API and convert it into Excel and CSV format

In [ ]:
# Import modules

from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)
from google.oauth2 import service_account
from google.protobuf.json_format import MessageToDict
import pandas as pd
import os


## Global Variables, property ID, creds

In [3]:
## Set up global variables

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'key.json'
property_id = '445050149'

client = BetaAnalyticsDataClient()

## Making first request for Geographic details

In [154]:
# Get the data from the API

request = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="country"),
        Dimension(name="region"),
        Dimension(name="city"),
        Dimension(name="date"),
        Dimension(name="newVsReturning"),
        Dimension(name="platform")
    ],
    metrics=[
        Metric(name="averageSessionDuration"),
        Metric(name="eventCount"),
        Metric(name="totalUsers")
    ],
    date_ranges=[DateRange(start_date="2020-03-31", end_date="today")],
    )

response = client.run_report(request)

## Making second request for Tech details

In [167]:
# Get the data from the API

request2 = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="browser"),
        Dimension(name="deviceCategory"),
        Dimension(name="deviceModel"),
        Dimension(name="pageTitle"),
        Dimension(name="newVsReturning"),
        Dimension(name="operatingSystem"),
        Dimension(name="operatingSystemVersion"),
        Dimension(name="sessionSource")
    ],
    metrics=[
        Metric(name="activeUsers"),
        Metric(name="sessions")],
    date_ranges=[DateRange(start_date="2020-03-31", end_date="today")],
    )

response2 = client.run_report(request2)

## Making third request for Event names and their count

In [156]:
# Get the data from the API

request3 = RunReportRequest(
    property=f"properties/{property_id}",
    dimensions=[
        Dimension(name="audienceId"),
        Dimension(name="eventName"),
        Dimension(name="platform")
    ],
    metrics=[
        Metric(name="eventCount")
    ],
    date_ranges=[DateRange(start_date="2020-03-31", end_date="today")],
    )

response3 = client.run_report(request3)

## Function for converting the received data into a formatted table

In [126]:
# Turn the raw data into a Table

def ga4_result_to_df(response):
    """Original: print_run_report_response: Prints results of a runReport call. v2.1 changed by Bram to create DataFrame"""
    result_dict = {}
    for dimensionHeader in response.dimension_headers:
        result_dict[dimensionHeader.name] = []
    for metricHeader in response.metric_headers:
        result_dict[metricHeader.name] = []
    for rowIdx, row in enumerate(response.rows):
        for i, dimension_value in enumerate(row.dimension_values):
            dimension_name = response.dimension_headers[i].name
            result_dict[dimension_name].append(dimension_value.value)
        for i, metric_value in enumerate(row.metric_values):
            metric_name = response.metric_headers[i].name
            result_dict[metric_name].append(metric_value.value)
    return pd.DataFrame(result_dict)



## Location Details

In [159]:
df = ga4_result_to_df(response)
df

,audienceId,country,region,city,date,newVsReturning,platform,averageSessionDuration,eventCount,totalUsers
0,8234410748,India,Maharashtra,Pune,20240607,new,web,1779.660788,35,1
1,8234410748,India,Maharashtra,Pune,20240607,returning,web,264.75829933333335,55,1


## Formatting the output (Date and Time)

In [160]:

# Convert the sessionDuration column to floats and then to integers
df['averageSessionDuration'] = df['averageSessionDuration'].astype(float).astype(int)


# Function to convert seconds to HH:MM format
def convert_seconds_to_hhmm(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f'{hours:02}:{minutes:02}'

# Apply the function to the sessionDuration column
df['avgSessionDuration_HHMM'] = df['averageSessionDuration'].apply(convert_seconds_to_hhmm)

# Drop the original sessionDuration column
df.drop('averageSessionDuration', axis=1, inplace=True)




# Convert the date column to datetime format
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

# Reformat the date to DD-MM-YYYY
df['formatted_date'] = df['date'].dt.strftime('%d-%m-%Y')

df.drop('date', axis=1, inplace=True)

df

,audienceId,country,region,city,newVsReturning,platform,eventCount,totalUsers,avgSessionDuration_HHMM,formatted_date
0,8234410748,India,Maharashtra,Pune,new,web,35,1,00:29,07-06-2024
1,8234410748,India,Maharashtra,Pune,returning,web,55,1,00:04,07-06-2024


## Tech Details

In [168]:
df2 = ga4_result_to_df(response2)
df2

,audienceId,browser,deviceCategory,deviceModel,pageTitle,newVsReturning,operatingSystem,operatingSystemVersion,sessionSource,activeUsers,sessions
0,8234410748,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,new,Windows,11,(direct),1,1
1,8234410748,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,returning,Windows,11,(direct),1,3
2,8234410748,Chrome,desktop,(not set),Talk to your data ‹ Testing — WordPress,returning,Windows,11,(direct),1,2


## Event Details

In [162]:
df3 = ga4_result_to_df(response3)
df3

,audienceId,eventName,platform,eventCount
0,8234410748,page_view,web,29
1,8234410748,scroll,web,28
2,8234410748,user_engagement,web,26
3,8234410748,session_start,web,4
4,8234410748,first_visit,web,1
5,8234410748,form_start,web,1
6,8234410748,form_submit,web,1


## Merging data into one table

In [169]:
# Merge DataFrames on audienceId and newVsReturning columns
merged_df = pd.merge(df, df2, on=['audienceId', 'newVsReturning'], how='inner')
merged_df

,audienceId,country,region,city,newVsReturning,platform,eventCount,totalUsers,avgSessionDuration_HHMM,formatted_date,browser,deviceCategory,deviceModel,pageTitle,operatingSystem,operatingSystemVersion,sessionSource,activeUsers,sessions
0,8234410748,India,Maharashtra,Pune,new,web,35,1,00:29,07-06-2024,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,Windows,11,(direct),1,1
1,8234410748,India,Maharashtra,Pune,returning,web,55,1,00:04,07-06-2024,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,Windows,11,(direct),1,3
2,8234410748,India,Maharashtra,Pune,returning,web,55,1,00:04,07-06-2024,Chrome,desktop,(not set),Talk to your data ‹ Testing — WordPress,Windows,11,(direct),1,2


In [171]:
# Rename columns
new_column_names = {
    'audienceId': 'Audience ID',
    'country': 'Country',
    'region': 'Region',
    'city': 'City',
    'newVsReturning': 'New/Returning',
    'platform':'Platform',
    'eventCount': 'Event Count',
    'totalUsers': 'Total Users',
    'avgSessionDuration_HHMM': 'Avg Session Duration',
    'formatted_date': 'Date',
    'browser': 'Browser',
    'deviceCategory': 'Device Category',
    'deviceModel': 'Device Model',
    'pageTitle': 'Page Title',
    'operatingSystem': 'OS',
    'operatingSystemVersion': 'OS Version',
    'sessionSource': 'Session Source',
    'activeUsers': 'Active Users',
    'sessions': 'Sessions',
}
merged_df.rename(columns=new_column_names, inplace=True)
merged_df

,Audience ID,Country,Region,City,New/Returning,Platform,Event Count,Total Users,Avg Session Duration,Date,Browser,Device Category,Device Model,Page Title,OS,OS Version,Session Source,Active Users,Sessions
0,8234410748,India,Maharashtra,Pune,new,web,35,1,00:29,07-06-2024,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,Windows,11,(direct),1,1
1,8234410748,India,Maharashtra,Pune,returning,web,55,1,00:04,07-06-2024,Chrome,desktop,(not set),Connections ‹ Testing — WordPress,Windows,11,(direct),1,3
2,8234410748,India,Maharashtra,Pune,returning,web,55,1,00:04,07-06-2024,Chrome,desktop,(not set),Talk to your data ‹ Testing — WordPress,Windows,11,(direct),1,2


In [177]:
merged_df.to_csv('GA4_python_output.csv')
df3.to_csv('Event_details.csv')

In [178]:

## Export to Excel
# Export to Excel without the index column
merged_df.to_excel('GA4_python_output.xlsx', sheet_name='GA4_report', index=False, engine='xlsxwriter')
df3.to_excel('Event details.xlsx', sheet_name='Event report', index=False, engine='xlsxwriter')
